# `c06_gr` — Graduation Rates at 150 Percent of Normal Time

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `GR2023`, `DRVGR2023` |
| Reference period | Status as of August 31, 2023 for the 2017 entering cohort at 4-year institutions and the 2020 entering cohort at 2-year institutions |
| Curated grain | `UNITID` x `GRTYPE` |
| Output | `data/curated/c06_gr.parquet` |

The cohort year differs by institution level within this one file: 2017 entering for 4-year institutions, 2020 entering for 2-year institutions, both measured as of August 31, 2023. A panel that treats GR2023 as a single cohort year is wrong for one of the two sectors.

> **Pitfall.** GRTYPE encodes cohort mechanics, not demographics: the revised cohort, exclusions, the adjusted cohort, completers, and transfer-outs are separate rows. Compute rates as completers over adjusted cohort and retain numerator and denominator as separate columns, so the beta-binomial model in Notebook 04 remains possible. A pre-divided rate throws away the sample size that model needs.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c06_gr"
TABLES = ['GR2023', 'DRVGR2023']
GRAIN = ['UNITID', 'GRTYPE']
REFERENCE_PERIOD = 'Status as of August 31, 2023 for the 2017 entering cohort at 4-year institutions and the 2020 entering cohort at 2-year institutions'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,GR2023,913058,f00c7140d4dbbb056043689aa82835eec5f6101b907b04...,2026-09-24T17:19:24+00:00
1,DRVGR2023,123219,884ef3518a426926b447aff48a78f2dde7b5df6b2ef6ba...,2026-09-24T17:19:24+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'cohort year 2017',
    table=TABLES[0],
)
print(intro[:600])

File documentation for graduation rate data, 150% of normal time to complete - cohort year 2017 (4-year) and cohort year 2020 (2-year) institutions:  2023
(Provisional release)
Filename GR2023
Overview This file contains the graduation rate status as of August 31, 2023 for the cohort of full-time, first-time degree/certificate-seeking undergraduates in both four year and two year institutions. Data for four year institutions include the number of bachelor degree-seeking students who were enrolled in 2017, the number of bachelor degree seeking students who completed any degree/certificate withi


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

36 variables documented, 83 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,GRTYPE,Cohort data
2,CHRTSTAT,Graduation rate status in cohort
3,SECTION,Section of survey form
4,COHORT,Cohort
5,LINE,Original line number of survey form
6,GRTOTLT,Grand total
7,GRTOTLM,Total men
8,GRTOTLW,Total women
9,GRAIANT,American Indian or Alaska Native total


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'GRTYPE', 'CHRTSTAT', 'SECTION', 'COHORT', 'GRTOTLT', 'GRTOTLM', 'GRTOTLW']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (51368, 66)
schema: unchanged | added [] | removed []


,UNITID,GRTYPE,CHRTSTAT,SECTION,COHORT,GRTOTLT,GRTOTLM,GRTOTLW
0,100654,1,10,1,1,1286,514.0,772.0
1,100654,2,12,1,1,1284,513.0,771.0
2,100654,3,13,1,1,369,112.0,257.0
3,100654,4,20,1,1,393,130.0,263.0
4,100654,6,10,2,2,1286,514.0,772.0


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['GRTYPE', 'CHRTSTAT']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 6 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
2,XGRTOTLM,R,46307,0.9015
3,XGRTOTLM,A,3751,0.0730
4,XGRTOTLM,Z,1305,0.0254
5,XGRTOTLM,P,5,0.0001
0,XGRTOTLT,R,51360,0.9998
1,XGRTOTLT,P,8,0.0002
6,XGRTOTLW,R,45901,0.8936
7,XGRTOTLW,A,3751,0.0730
8,XGRTOTLW,Z,1711,0.0333
9,XGRTOTLW,P,5,0.0001


Columns under 90% reported — interpret with care:


column
XGRTOTLW    0.8936
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['GRTYPE', 'CHRTSTAT']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,GRTYPE,CHRTSTAT,GRTYPE_LABEL,CHRTSTAT_LABEL
0,1,10,4-year institutions revised cohort,Revised cohort
1,2,12,"4-year institutions, Adjusted cohort (revised ...",Adjusted cohort (revised cohort minus exclusions)
2,3,13,"4-year institutions, Completers within 150% of...",Completers within 150% of normal time
3,4,20,"4-year institutions, Transfer-out students",Transfer-out students
4,6,10,Bachelor's or equiv subcohort (4-yr institution),Revised cohort
5,7,11,Bachelor's or equiv subcohort (4-yr institutio...,Exclusions
6,8,12,Bachelor's or equiv subcohort (4-yr institutio...,Adjusted cohort (revised cohort minus exclusions)
7,9,13,Bachelor's or equiv subcohort (4-yr institutio...,Completers within 150% of normal time
8,12,16,Bachelor's or equiv subcohort (4-yr institutio...,Completers of bachelor's or equivalent degrees...
9,13,17,Bachelor's or equiv subcohort (4-yr institutio...,Completers of bachelor's or equivalent degrees...


## 9. Reshape to the declared grain

Target grain: `UNITID` x `GRTYPE`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID', 'GRTYPE'] -> 51,368 rows, 0 duplicated


,UNITID,GRTYPE,CHRTSTAT,SECTION,COHORT,GRTOTLT,GRTOTLM,GRTOTLW,GRTYPE_LABEL,CHRTSTAT_LABEL
0,100654,1,10,1.0,1.0,1286.0,514.0,772.0,4-year institutions revised cohort,Revised cohort
1,100654,2,12,1.0,1.0,1284.0,513.0,771.0,"4-year institutions, Adjusted cohort (revised ...",Adjusted cohort (revised cohort minus exclusions)
2,100654,3,13,1.0,1.0,369.0,112.0,257.0,"4-year institutions, Completers within 150% of...",Completers within 150% of normal time
3,100654,4,20,1.0,1.0,393.0,130.0,263.0,"4-year institutions, Transfer-out students",Transfer-out students
4,100654,6,10,2.0,2.0,1286.0,514.0,772.0,Bachelor's or equiv subcohort (4-yr institution),Revised cohort


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID', 'GRTYPE'),
    iu.in_range('GRTOTLT', 0, None),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,"unique_key(UNITID,GRTYPE)",pass,0,0.0,Declared grain must be unique
1,"in_range(GRTOTLT,0,None)",pass,0,0.0,Value plausibility bound


PASSED


Report(table='c06_gr', rows=51368, results=[{'name': 'unique_key(UNITID,GRTYPE)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(GRTOTLT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}], generated_utc='2026-09-24T17:19:24+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='GRTYPE encodes cohort mechanics, not demographics: the revised cohort, exclusions, the adjusted cohort, completers, and transfer-outs are separate rows. Compute rates as completers over adjusted cohort and retain numerator and denominator as separate columns, so the beta-binomial model in Notebook 04 remains possible. A pre-divided rate throws away the sample size that model needs.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c06_gr.parquet (51,368 rows x 10 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. GRTYPE encodes cohort mechanics, not demographics: the revised cohort, exclusions, the adjusted cohort, completers, and transfer-outs are separate rows. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.